# 足球接球预测分析 - 2022世界杯版本

本notebook展示了使用图注意力网络(GAT)分析足球接球预测的交互式可视化工具。

**适配版本**: 2022 FIFA世界杯数据集

## 概述

- **数据源**: 2022世界杯追踪数据
- **模型**: 图注意力网络(GAT)用于接球预测
- **可视化**: 交互式球场可视化
- **功能**: 防守影响力分析、注意力权重可视化、球员交互分析

## 1. 导入必要的库

In [ ]:
# 标准库导入
import bz2
import json
import os
import pickle
import gc

# 数据处理导入
import pandas as pd
import numpy as np

# PyTorch导入
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool, GENConv, SAGEConv, GATv2Conv
from torch_geometric.data import Data, DataLoader

# Scikit-learn导入
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MinMaxScaler

# 自定义模块导入
import convert_tracking as ct
import plot_functions as pf
import create_graph as cg
import scale_graph as sg
import visualisation
from GNNs.custom_GAT import myGATv2Conv
import GNNs.model_training as mt
import GNNs.convert_data as cd
from GNNs.GNN import ReceptionPredictionGNN
from GNNs.GAT import GATReceptionPredictor

print("✓ 所有库导入成功")

## 2. 配置参数和数据路径

设置要分析的比赛ID和相关路径。

In [ ]:
# 配置参数
# 可以选择以下任一比赛：
# '10517' - 决赛: 阿根廷 vs 法国
# '3857' - 季军赛: 克罗地亚 vs 摩洛哥
# 或其他任何已处理的比赛ID

GAME_ID = '10517'  # 默认使用决赛数据
DATA_DIR = f'Data/{GAME_ID}'
XT_GRID_PATH = 'xT_grid.csv'

# 模型路径 - 使用训练好的模型
MODEL_PATH = f'results/model_final_{GAME_ID}.pth'
SCALER_PATH = f'results/scaler_final_{GAME_ID}.pkl'

# 检查文件是否存在
if not os.path.exists(DATA_DIR):
    print(f"警告: 数据目录不存在: {DATA_DIR}")
    print(f"请先运行 convert_tracking.py 处理比赛 {GAME_ID}")
else:
    print(f"✓ 数据目录存在: {DATA_DIR}")

if not os.path.exists(MODEL_PATH):
    print(f"警告: 模型文件不存在: {MODEL_PATH}")
    print(f"请先运行 RunGATModel_2022WC.ipynb 或 test_run_gat_model_final.ipynb 训练模型")
else:
    print(f"✓ 模型文件存在: {MODEL_PATH}")

print(f"\n配置完成:")
print(f"  比赛ID: {GAME_ID}")
print(f"  数据目录: {DATA_DIR}")
print(f"  模型路径: {MODEL_PATH}")

## 3. 加载Expected Threat (xT)网格数据

In [ ]:
# 加载Expected Threat (xT)网格
if os.path.exists(XT_GRID_PATH):
    xT_grid = pd.read_csv(XT_GRID_PATH, header=None)
    print(f"✓ xT网格加载成功")
    print(f"  网格形状: {xT_grid.shape}")
else:
    print(f"警告: xT网格文件不存在: {XT_GRID_PATH}")
    print("创建默认xT网格...")
    # 创建一个简单的默认xT网格（从防守端到进攻端递增）
    xT_grid = pd.DataFrame(np.linspace(0, 1, 12*8).reshape(12, 8))
    print(f"✓ 默认xT网格创建完成")

## 4. 加载比赛元数据

In [ ]:
# 加载比赛元数据
print("加载比赛元数据...")
(
    home_team_id, away_team_id, home_team_name, away_team_name, 
    home_team_start_left, rosters_for_game_home, rosters_for_game_away,
    roster_game_home_name_dict, roster_game_home_team_name_dict, roster_game_home_pos_dict,
    roster_game_away_name_dict, roster_game_away_team_name_dict, roster_game_away_pos_dict,
    pitch_x_adjustment, pitch_y_adjustment
) = ct.get_metadata(GAME_ID)

print(f"\n✓ 元数据加载成功")
print(f"  比赛: {home_team_name} vs {away_team_name}")
print(f"  主队开球方向: {'左' if home_team_start_left else '右'}")
print(f"  主队球员数: {len(rosters_for_game_home)}")
print(f"  客队球员数: {len(rosters_for_game_away)}")

## 5. 加载处理后的比赛数据

In [ ]:
# 加载处理后的数据框
print("加载处理后的数据...")

balls_df = pd.read_csv(f'{DATA_DIR}/balls_{GAME_ID}.csv')
events_df = pd.read_csv(f'{DATA_DIR}/events_{GAME_ID}.csv')
players_df = pd.read_csv(f'{DATA_DIR}/players_{GAME_ID}.csv')

# 计算球的速度（如果还没有）
if 'velocity_x' not in balls_df.columns:
    balls_df = ct.calculate_ball_velocities(balls_df)

print(f"\n✓ 数据加载成功")
print(f"  球追踪点: {len(balls_df)}")
print(f"  事件数: {len(events_df)}")
print(f"  球员追踪点: {len(players_df)}")
print(f"  唯一球员数: {players_df['playerName'].nunique()}")

## 6. 创建图数据集

为每个事件帧创建图结构。这可能需要几分钟时间。

In [ ]:
# 创建图数据集
print("为事件帧创建图...")
print("注意: 这可能需要几分钟时间\n")

graphs = []
event_frames = events_df['frameNum'].values

print(f"总共 {len(event_frames)} 个事件帧需要处理")

for i, frameNum in enumerate(event_frames):
    if i % 200 == 0:  # 进度指示器
        print(f"  处理进度: {i+1}/{len(event_frames)} ({100*(i+1)/len(event_frames):.1f}%)")
    
    G = cg.create_normalized_graph_directed(players_df, balls_df, events_df, frameNum, home_team_name)
    if G is not None:
        graphs.append(G)

print(f"\n✓ 图创建完成")
print(f"  成功创建: {len(graphs)} 个图")
print(f"  失败: {len(event_frames) - len(graphs)} 个图")

## 7. 加载或创建缩放后的图

尝试加载预先缩放的图，如果不存在则创建新的。

In [ ]:
# 尝试加载预缩放的图
scaled_graphs_path = f'results/scaled_graphs_2022WC.pkl'

try:
    print(f"尝试加载预缩放的图: {scaled_graphs_path}")
    with open(scaled_graphs_path, 'rb') as f:
        saved_data = pickle.load(f)
        
    # 检查是否包含当前比赛的数据
    if 'game_ids' in saved_data and GAME_ID in saved_data['game_ids']:
        # 提取当前比赛的图
        game_indices = [i for i, gid in enumerate(saved_data['game_ids']) if gid == GAME_ID]
        scaled_graphs = [saved_data['scaled_graphs'][i] for i in game_indices]
        print(f"✓ 加载了 {len(scaled_graphs)} 个预缩放的图（比赛 {GAME_ID}）")
    else:
        raise FileNotFoundError("预缩放文件中没有当前比赛的数据")
        
except FileNotFoundError as e:
    print(f"未找到预缩放的图或不包含当前比赛数据")
    print("创建新的缩放图...\n")
    
    # 创建图缩放器并拟合数据
    graph_scaler = sg.GraphFeatureScaler()
    graph_scaler.fit(graphs)
    print("✓ 缩放器已拟合")
    
    # 缩放图
    scaled_graphs = [graph_scaler.transform_graph(g) for g in graphs]
    print(f"✓ 创建了 {len(scaled_graphs)} 个缩放图")
    
    # 保存缩放器供后续使用
    os.makedirs('results', exist_ok=True)
    with open(SCALER_PATH, 'wb') as f:
        pickle.dump(graph_scaler, f)
    print(f"✓ 缩放器已保存到: {SCALER_PATH}")

## 8. 模型定义和加载

定义GAT模型架构并加载训练好的权重。

In [ ]:
# 初始化并加载训练好的模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 模型参数配置
MODEL_CONFIG = {
    'num_node_features': 15,
    'num_edge_features': 6,
    'hidden_channels': 32,
    'edge_hidden_channels': 16,
    'num_heads': 16
}

# 创建模型实例
loaded_model = GATReceptionPredictor(**MODEL_CONFIG).to(device)

# 加载模型权重
if os.path.exists(MODEL_PATH):
    model_state = torch.load(MODEL_PATH, map_location=device)
    loaded_model.load_state_dict(model_state)
    print(f"✓ 模型加载成功: {MODEL_PATH}")
else:
    print(f"警告: 模型文件不存在: {MODEL_PATH}")
    print("将使用未训练的模型（结果可能不准确）")

# 显示模型信息
total_params = sum(p.numel() for p in loaded_model.parameters())
trainable_params = sum(p.numel() for p in loaded_model.parameters() if p.requires_grad)
print(f"\n模型信息:")
print(f"  总参数数: {total_params:,}")
print(f"  可训练参数: {trainable_params:,}")
print(f"  隐藏通道数: {MODEL_CONFIG['hidden_channels']}")
print(f"  注意力头数: {MODEL_CONFIG['num_heads']}")

## 9. 交互式可视化

启动交互式可视化界面。这将创建一个交互式工具，允许您：

- **浏览不同的比赛帧**
- **分析防守球员的影响力**
- **可视化球员之间的注意力权重**
- **比较不同的比赛状态**
- **拖动球员位置查看影响**

### 使用说明：

1. **Graph Index**: 选择要查看的帧
2. **Defender**: 选择要分析的防守球员
3. **Show Defender Influence**: 显示该防守球员对进攻球员的影响
4. **Show Defender Performances**: 显示所有防守球员的表现值
5. **Player**: 选择要查看边连接的球员
6. **Show Player Edges**: 显示该球员到对方球员的连接
7. **Show Attention Weights**: 显示注意力权重（线条粗细表示权重大小）

### 颜色说明：
- **红色**: 进攻球员
- **蓝色**: 防守球员
- **绿色**: 被选中的防守球员
- **黑色**: 球

In [ ]:
# 启动交互式可视化
print("启动交互式可视化...")
print("\n提示: 可视化将在下方显示")
print("      使用控件探索不同的比赛状态和分析选项\n")

visualisation.create_simple_visualization(loaded_model, graphs, scaled_graphs, xT_grid)

## 10. 示例分析：预测接球概率

对特定帧进行详细的接球概率分析。

In [ ]:
# 选择一个示例图进行分析
example_graph_idx = min(40, len(scaled_graphs) - 1)

print(f"分析图 {example_graph_idx}...")
results_df, attention_analysis = visualisation.predict_reception_probabilities(
    loaded_model,
    scaled_graphs[example_graph_idx],
    head_indexes=range(2)  # 使用前2个注意力头
)

print(f"\n✓ 预测完成\n")
print("="*60)
print("接球概率预测（前10名）:")
print("="*60)
print(results_df.head(10).to_string(index=False))

# 显示最高概率球员的注意力权重
if len(results_df) > 0:
    top_player = results_df['player_name'].iloc[0]
    top_prob = results_df['reception_probability_AT'].iloc[0]
    
    print(f"\n{'='*60}")
    print(f"{top_player} 的注意力权重分析")
    print(f"接球概率: {top_prob:.4f}")
    print(f"{'='*60}")
    
    if top_player in attention_analysis:
        print("\n来自其他球员的注意力（前10）:")
        print(f"{'排名':<6} {'球员':<20} {'注意力权重':<15}")
        print("-" * 45)
        for i, attn in enumerate(attention_analysis[top_player][:10]):
            print(f"{i+1:<6} {attn['source_player']:<20} {attn['attention_weight']:.6f}")

## 11. 可视化示例图

在球场上可视化示例图的结构。

In [ ]:
# 可视化示例图
if len(graphs) > example_graph_idx:
    print(f"可视化图 {example_graph_idx}...")
    
    G = graphs[example_graph_idx]
    frame_num = event_frames[example_graph_idx] if example_graph_idx < len(event_frames) else 0
    
    # 基本图可视化
    fig, ax = pf.visualize_graph_on_pitch_AT(
        G, 
        sample_timestamp=frame_num,
        ball=True,
        edges=True
    )
    
    print(f"\n图统计:")
    print(f"  节点数: {G.number_of_nodes()}")
    print(f"  边数: {G.number_of_edges()}")
    print(f"  帧编号: {frame_num}")

## 总结

本notebook完成了以下工作：

1. ✓ 加载2022世界杯比赛数据
2. ✓ 创建图数据集
3. ✓ 加载训练好的GAT模型
4. ✓ 启动交互式可视化工具
5. ✓ 进行接球概率预测和注意力分析

### 下一步探索：

- 使用交互式工具分析不同的防守策略
- 比较不同球员的注意力模式
- 分析关键时刻的球员交互
- 评估防守球员的整体表现

### 相关文件：

- `RunGATModel_2022WC.ipynb` - 训练GAT模型
- `test_run_gat_model_final.ipynb` - 快速测试模型
- `player_eval.py` - 球员评估工具
- `Experiments.ipynb` - 深入实验分析